<a href="https://colab.research.google.com/github/sudhars97/Ecommerce-taxonomy-pipeline/blob/main/notebooks/03_semantic_search_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install vector database and embedding libraries
!pip install chromadb sentence-transformers langchain-google-genai pandas -q

import os
import pandas as pd
import chromadb
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# 2. Securely load API Key for the RAG generation phase
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# 3. Create a clean mock catalog to index
print("Initializing Product Catalog...")
products = [
    {"id": "item_1", "title": "Nike Men's Running Shoes", "desc": "Lightweight breathable running shoes for daily jogging on pavement."},
    {"id": "item_2", "title": "Arctic Explorer Winter Coat", "desc": "Heavy insulated waterproof coat designed for extreme cold and snow."},
    {"id": "item_3", "title": "Ray-Ban Aviator Sunglasses", "desc": "Classic polarized sunglasses with UV protection for sunny days."},
    {"id": "item_4", "title": "Generic Mens Casual Shorts", "desc": "Comfortable cotton shorts, perfect for hot weather and beach trips."},
    {"id": "item_5", "title": "Leather Hiking Boots", "desc": "Durable high-ankle boots with deep tread for rough terrain and mountains."},
]

# 4. Initialize ChromaDB and create a Vector Store
# By default, Chroma uses the 'all-MiniLM-L6-v2' sentence-transformer model to create embeddings locally.
print("Embedding catalog into Vector Database...")
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="ecommerce_catalog")

# Add products to the vector database (this automatically creates the embeddings)
collection.add(
    documents=[p['desc'] for p in products],
    metadatas=[{"title": p['title']} for p in products],
    ids=[p['id'] for p in products]
)

# 5. Perform a Semantic Search
# Notice the query doesn't contain the words "shorts" or "sunglasses"
user_query = "I'm going on a tropical beach vacation, what should I pack?"
print(f"\nUSER QUERY: '{user_query}'\n")

# Retrieve the top 2 most semantically relevant products
results = collection.query(
    query_texts=[user_query],
    n_results=2
)

# Extract the retrieved items to feed into our LLM
retrieved_titles = [meta['title'] for meta in results['metadatas'][0]]
retrieved_descriptions = results['documents'][0]
context = "\n".join([f"- {t}: {d}" for t, d in zip(retrieved_titles, retrieved_descriptions)])

print("VECTORS RETRIEVED (Semantic Match):")
print(context)
print("-" * 50)

# 6. RAG (Retrieval-Augmented Generation)
# We pass the retrieved database products to the LLM so it can answer the user's question accurately.
rag_prompt = PromptTemplate.from_template("""
You are a helpful eCommerce shopping assistant.
A customer has asked: "{user_query}"

Based ONLY on the following products retrieved from our catalog, recommend them to the user and explain why they fit their needs.

Catalog Results:
{context}

Response:
""")

print("Generating LLM Response (RAG)...\n")
formatted_prompt = rag_prompt.format(user_query=user_query, context=context)
response = llm.invoke(formatted_prompt)

# Safely extract text (handling LangChain's list formatting if necessary)
raw_content = response.content
final_answer = raw_content[0].get('text', '') if isinstance(raw_content, list) else str(raw_content)

print(final_answer.strip())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 42.9MiB/s]



USER QUERY: 'I'm going on a tropical beach vacation, what should I pack?'



VECTORS RETRIEVED (Semantic Match):
- Generic Mens Casual Shorts: Comfortable cotton shorts, perfect for hot weather and beach trips.
- Ray-Ban Aviator Sunglasses: Classic polarized sunglasses with UV protection for sunny days.
--------------------------------------------------
Generating LLM Response (RAG)...

Here are some essential items from our catalog to pack for your tropical beach vacation:

1. **Generic Mens Casual Shorts**: These comfortable cotton shorts are perfect for hot weather and keeping cool during your beach trips. 
2. **Ray-Ban Aviator Sunglasses**: A classic choice for sunny days, offering UV protection and polarized lenses to protect your eyes while relaxing by the water. 

Let me know if you would like to add either of these to your cart!
